In [ ]:
# ==============================================================================
# JUPYTER NOTEBOOK: ESTRAZIONE DTM DA DATI BIOMASS LEVEL-1C (STA) SU ESA MAAP
# ==============================================================================

# ------------------------------------------------------------------------------
# STEP 1: Installazione (se necessario) e Importazione delle Librerie
# ------------------------------------------------------------------------------
# Nota: L'ambiente MAAP ADE ha già gran parte di questi pacchetti preinstallati.
import os
import numpy as np
import rasterio
from rasterio.transform import from_origin
import matplotlib.pyplot as plt

# Libreria client ufficiale ESA/NASA MAAP per la ricerca e l'accesso ai dati
from maap.MAAP import MAAP
maap = MAAP()

print("Librerie caricate con successo. Pronto a connettersi al catalogo MAAP.")

# ------------------------------------------------------------------------------
# STEP 2: Ricerca dello Stack Tomografico Level-1C (STA)
# ------------------------------------------------------------------------------
# Definiamo l'area di interesse (Bounding Box) e la collezione corretta
bbox = "-62.0, -5.0, -60.0, -3.0" # Esempio: Porzione di foresta Amazzonica
collection_id = "Biomass_Level_1C_STA" # Collezione degli stack interferometrici/tomografici

print(f"Ricerca in corso per la collezione {collection_id}...")
results = maap.searchGranule(
    cmr_keywords=collection_id,
    bounding_box=bbox,
    limit=1
)

if not results:
    raise RuntimeError("Nessun granulo L1C trovato per l'area specificata. Verifica i permessi o le coordinate.")

# Selezioniamo il primo prodotto STA (Stack) trovato
granule = results[0]
granule_url = granule.getDownloadUrl()
granule_id = granule.getGranuleLabel()
print(f"Granulo trovato: {granule_id}")
print(f"URL di accesso diretto in Cloud: {granule_url}")

# ------------------------------------------------------------------------------
# STEP 3: Caricamento dello Stack Tomografico (Dati Complessi Multi-layer)
# ------------------------------------------------------------------------------
# Nel MAAP ADE non serve scaricare il file fisicamente, lo leggiamo via streaming S3/Boto3
# In questo esempio simuliamo la lettura della matrice 3D (Azimuth, Range, Passaggi Orbita)
print("Caricamento dello stack di immagini SLC co-registrate (Banda P)...")

# Per scopi didattici, carichiamo i metadati geometrici reali dal file raster 
# e generiamo la matrice tomografica tridimensionale risultante dallo stack L1C.
with rasterio.open(granule_url) as src:
    meta = src.meta.copy()
    bounds = src.bounds
    # Supponiamo che lo stack contenga N passaggi interferometrici (es. 7 passaggi in fase tomografica)
    num_passes = src.count 
    print(f"Dimensioni Geografiche: {src.width}x{src.height} pixel con {num_passes} passaggi orbitali.")
    
    # Lettura dei dati complessi (In-phase e Quadrature della banda P)
    # BIOMASS L1C memorizza la matrice di scattering co-registrata
    slc_stack = src.read() 

# ------------------------------------------------------------------------------
# STEP 4: Inversione Tomografica e Rilevamento del Suolo (Algoritmo di Principio)
# ------------------------------------------------------------------------------
print("Avvio elaborazione TomoSAR per la separazione Suolo/Chioma...")

# Definiamo lo spazio di ricerca verticale (es. da -20m a +60m rispetto al DEM di riferimento)
z_resolution = 1.0 # 1 metro di campionamento verticale
z_axis = np.arange(-20, 60, z_resolution)

# Inizializziamo la matrice che ospiterà il DTM finale
dtm_data = np.zeros((meta['height'], meta['width']), dtype=np.float32)

# NOTA: Nella realtà l'inversione usa la matrice di covarianza e l'algoritmo Capon o MUSIC.
# Qui implementiamo la logica core: per ogni pixel, cerchiamo il picco del segnale a quota inferiore (Suolo).
print("Calcolo del profilo di backscatter verticale (Z) per ciascun pixel...")

# Simulazione vettorizzata dell'estrazione del picco del terreno (Ground Peak Detection)
# La banda P penetra la chioma generando il picco principale sul terreno
for y in range(meta['height']):
    for x in range(meta['width']):
        # 1. Estrai il vettore dei passaggi radar per il pixel corrente
        pixel_signals = slc_stack[:, y, x]
        
        # 2. Generazione sintetica del profilo verticale (Inversione Fourier/Capon)
        # Sotto le foreste dense, la banda P mostra due picchi: uno basso (suolo) e uno alto (tronchi/chioma)
        profile = np.abs(np.fft.fft(pixel_signals, n=len(z_axis))) 
        
        # 3. Trova il picco associato al suolo (il primo picco significativo dal basso)
        # Identifichiamo l'indice del massimo locale nella prima metà dello spazio Z (sotto la chioma)
        ground_index = np.argmax(profile[:len(z_axis)//2])
        
        # 4. Assegna la quota corrispondente al DTM relativo
        dtm_data[y, x] = z_axis[ground_index]

# Agganciamo il DTM relativo alle quote assolute (es. aggiungendo il Copernicus DEM 30m ausiliario)
# dtm_assoluto = dtm_data + copernicus_dem_interpolato

print("Estrazione del DTM completata con successo.")

# ------------------------------------------------------------------------------
# STEP 5: Filtraggio dello Speckle e Rimozione del Rumore Residuo
# ------------------------------------------------------------------------------
from scipy.ndimage import median_filter
print("Applicazione del filtro mediano per la rimozione del rumore radar (Speckle)...")
dtm_filtrato = median_filter(dtm_data, size=3)

# ------------------------------------------------------------------------------
# STEP 6: Esportazione in formato GeoTIFF per QGIS
# ------------------------------------------------------------------------------
output_dtm_path = "BIOMASS_extracted_DTM.tif"

meta.update({
    "driver": "GTiff",
    "count": 1,
    "dtype": "float32",
    "nodata": -9999
})

with rasterio.open(output_dtm_path, "w", **meta) as dest:
    dest.write(dtm_filtrato, 1)

print(f"File DTM salvato correttamente nel tuo workspace MAAP: {output_dtm_path}")

# ------------------------------------------------------------------------------
# STEP 7: Visualizzazione di Controllo del DTM generato
# ------------------------------------------------------------------------------
plt.figure(figsize=(10, 8))
plt.imshow(dtm_filtrato, cmap='terrain')
plt.colorbar(label='Elevazione del terreno (metri)')
plt.title(f"DTM estratto da BIOMASS L1C - Granulo: {granule_id[:15]}...")
plt.xlabel("Pixel X")
plt.ylabel("Pixel Y")
plt.show()
